In [7]:
%matplotlib inline
import random
import torch
import inspect


In [8]:
class HyperParameters:
    def save_hyperparameters(self, ignore=[]):
        frame = inspect.currentframe().f_back
        _,_,_,local_vars=inspect.getargvalues(frame)
        self.hparams={k:v for k,v in local_vars.items()
                      if k not in set(ignore+['self']) and not k.startswith('_')}
        for k,v in self.hparams.items():
            setattr(self,k,v)

class DataModule(HyperParameters):
    def __init__(self, root='../data', num_workers=4):
        self.save_hyperparameters()

    def get_dataloader(self, train):
        raise NotImplementedError

    def train_dataloader(self):
        return self.get_dataloader(train=True)

    def val_dataloader(self):
        return self.get_dataloader(train=False)


In [9]:
class SyntheticRegressionData(DataModule):
    def __init__(self, w, b, noise=0.01, num_train=1000, num_val=1000, batch_size=32):
        super().__init__()
        self.save_hyperparameters()
        n = num_train + num_val
        self.X = torch.randn(n, len(w))
        noise = torch.randn(n, 1)*noise
        self.y = torch.matmul(self.X, w.reshape((-1,1))) + b + noise

data = SyntheticRegressionData(w=torch.tensor([2, -3.4]), b=4.2)
print('feature:', data.X)
print('target:', data.y)
print(len(data.X) == len(data.y))



feature: tensor([[-0.7004, -0.0067],
        [-0.7662, -0.0330],
        [-1.2527,  0.1907],
        ...,
        [ 0.9115, -0.8853],
        [-0.1293, -0.8662],
        [-1.0443,  0.9880]])
target: tensor([[ 2.8135],
        [ 2.7732],
        [ 1.0383],
        ...,
        [ 9.0371],
        [ 6.8958],
        [-1.2391]])
True


In [13]:
def add_to_class(Class):
    def wrapper(obj):
        setattr(Class, obj.__name__, obj)
    return wrapper

@add_to_class(SyntheticRegressionData)
def get_dataloader(self, train):
    if train:
        indices = list(range(self.num_train))
        random.shuffle(indices)
    else:
        indices = list(range(self.num_train, self.num_train + self.num_val))

    for i in range(0, len(indices), self.batch_size):
        batch_indices = torch.tensor(indices[i:i+self.batch_size])
        yield self.X[batch_indices], self.y[batch_indices]

X_train,y_train=next(iter(data.train_dataloader()))
print(X_train[:5], y_train[:5])

X_val,y_val=next(iter(data.val_dataloader()))
print(X_val[:5], y_val[:5])



tensor([[-1.5593, -1.6578],
        [-0.7741,  1.3088],
        [-0.3228, -1.6393],
        [ 3.2288, -0.9391],
        [ 1.2779,  0.7045]]) tensor([[ 6.7106],
        [-1.8000],
        [ 9.1152],
        [13.8504],
        [ 4.3544]])
tensor([[ 0.0224,  0.3741],
        [ 0.0768, -0.5920],
        [-0.4238,  0.4997],
        [-1.1487,  0.9292],
        [-0.2564, -0.9533]]) tensor([[ 2.9683],
        [ 6.3574],
        [ 1.6484],
        [-1.2647],
        [ 6.9506]])


In [14]:
@add_to_class(DataModule)
def get_tensorloader(self, tensors, train, indices=slice(0, None)):
    tensors = tuple(a[indices] for a in tensors)
    dataset = torch.utils.data.TensorDataset(*tensors)
    return torch.utils.data.DataLoader(dataset, self.batch_size, shuffle=train)

@add_to_class(SyntheticRegressionData)
def get_dataloader(self, train):
    i = slice(0, self.num_train) if train else slice(self.num_train, None)
    return self.get_tensorloader((self.X, self.y), train, i)

X_train,y_train=next(iter(data.train_dataloader()))
print(X_train[:5], y_train[:5])

X_val,y_val=next(iter(data.val_dataloader()))
print(X_val[:5], y_val[:5])

tensor([[-0.4016,  0.4979],
        [-2.4945, -0.0981],
        [ 0.4909, -0.1063],
        [ 0.9134,  0.0264],
        [ 1.5808,  0.8665]]) tensor([[ 1.7058],
        [-0.4437],
        [ 5.5561],
        [ 5.9374],
        [ 4.4170]])
tensor([[ 0.0224,  0.3741],
        [ 0.0768, -0.5920],
        [-0.4238,  0.4997],
        [-1.1487,  0.9292],
        [-0.2564, -0.9533]]) tensor([[ 2.9683],
        [ 6.3574],
        [ 1.6484],
        [-1.2647],
        [ 6.9506]])
